### Try an analysis

In [ ]:
import os
os.environ['PISA_RESOURCES'] = "../"
os.environ['PISA_RESOURCES'] += os.pathsep + "../../"
os.environ['PISA_RESOURCES'] += os.pathsep + "/data/ana/LE/"

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from uncertainties import unumpy as unp
import matplotlib.pyplot as plt
import pickle
import pisa
import copy
from scipy.special import erfcinv, erfc

from pisa.core.pipeline import Pipeline
from pisa.core.distribution_maker import DistributionMaker
from pisa.core.detectors import Detectors
from pisa import FTYPE, ureg
from pisa.utils.fileio import from_file, to_file
from pisa.core.map import MapSet
from pisa.analysis.analysis import Analysis
from pisa.analysis.analysis import update_param_values, update_param_values_detector

In [ ]:
params = {'legend.fontsize': 18,
          'figure.figsize': (9, 9*0.618),
          'axes.labelsize': 18,
          'axes.titlesize': 18,
          'xtick.labelsize': 18,
          'ytick.labelsize': 18}
plt.rcParams.update(params)

In [ ]:
def get_unc_med(sample):
    N = len(sample)
    n = (N-1)/2
    return np.std(sample)/np.sqrt(N) * np.sqrt(np.pi*N/(4*n))

def plot_med(sample, color, linewidth=1, label=''):
    med, unc_med = np.median(sample), get_unc_med(sample)
    plt.axvline(med, color=color, linewidth=linewidth, label=label)
    #plt.axvspan(med-unc_med, med+unc_med, alpha=0.5, color=color)
    
def calc_sens(nh, ih):
    return np.sqrt(2)*(nh+ih)/np.sqrt(8*ih)

In [ ]:
%%time
#template_maker_NO = DistributionMaker(["settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg"])
#template_maker_IO = DistributionMaker(["settings/pipeline/pipeline_upgrade_neutrinos_std_osc_IO.cfg"])
#template_maker_NO = DistributionMaker(["settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg"])
#template_maker_IO = DistributionMaker(["settings/pipeline/pipeline_oscnext_neutrinos_std_osc_IO.cfg"])


p1_nu_NO = Pipeline("settings/pipeline/pipeline_upgrade_neutrinos_std_osc_NO.cfg")
p1_nu_IO = Pipeline("settings/pipeline/pipeline_upgrade_neutrinos_std_osc_IO.cfg")
p1_mu = Pipeline("settings/pipeline/pipeline_upgrade_muons.cfg")
p2_nu_NO = Pipeline("settings/pipeline/pipeline_oscnext_neutrinos_std_osc_NO.cfg")
p2_nu_IO = Pipeline("settings/pipeline/pipeline_oscnext_neutrinos_std_osc_IO.cfg")
#p2_mu = Pipeline("settings/pipeline/pipeline_oscnext_muons.cfg")

shared_params = list(p1_nu_NO.params.free.names) #+ list(p1_mu.params.free.names)
shared_params.remove('icu_dom_eff')
template_maker_NO = Detectors([p1_nu_NO, p1_mu, p2_nu_NO], shared_params=shared_params)
template_maker_IO = Detectors([p1_nu_IO, p1_mu, p2_nu_IO], shared_params=shared_params)

In [ ]:
template_maker_NO.params.free

### Fits

In [ ]:
analysis = Analysis()
analysis.pprint = False

In [ ]:
# global minimizer
nlopt_settings = {
    "method": "nlopt",
    "method_kwargs": {
        "algorithm": "NLOPT_GN_CRS2_LM",
        "ftol_abs": 1e-3,
        "ftol_rel": 1e-3,
        # other options that can be set here: 
        # xtol_abs, xtol_rel, stopval, maxeval, maxtime
        # after maxtime seconds, stop and return best result so far
        "maxtime": 500
    },
    "local_fit_kwargs": None  # no further nesting available
}

# local minimizer
local_fit_minuit = {
    "method": "iminuit",
    "method_kwargs": {
        "errors": 0.1,
        "precision": 1e-14,  # default: double precision
        "tol": 1e-2,  # default: 0.1
        "run_simplex": False,
        "run_migrad": True
    },
    "local_fit_kwargs": None
}

# octant fit for local minimizer
fit_octant = {
    "method": "octants",
    "method_kwargs": {
        "angle": "theta23",
        "inflection_point": 45 * ureg.degrees,
    },
    "local_fit_kwargs": local_fit_minuit
}

# complete fit routine
fit_combi = {
    "method": "staged",
    "method_kwargs": None,
    "local_fit_kwargs":[
        nlopt_settings,
        fit_octant,
    ]
}

simple Asimov fits

In [ ]:
fake_data_nh = template_maker_NO.get_outputs(return_sum=True)

In [ ]:
%%time
#template_maker_IO.params.reset_free()
best_fit_info_nh = analysis.fit_recursively(
            data_dist=fake_data_nh,
            hypo_maker=template_maker_IO,
            metric=["mod_chi2"],
            external_priors_penalty=None,
            **fit_octant
        )

In [ ]:
nh_asi_ICU = best_fit_info_nh.detailed_metric_info[0]['mod_chi2']['maps']['total']
nh_asi_DC = best_fit_info_nh.detailed_metric_info[1]['mod_chi2']['maps']['total']
nh_prior = sum(best_fit_info_nh.detailed_metric_info[0]['mod_chi2']['priors'])

nh_asi = best_fit_info_nh.metric_val
print(nh_asi, np.sqrt(nh_asi))

In [ ]:
fake_data_ih = template_maker_IO.get_outputs(return_sum=True)

In [ ]:
%%time
#template_maker_NO.params.reset_free()
best_fit_info_ih = analysis.fit_recursively(
            data_dist=fake_data_ih,
            hypo_maker=template_maker_NO,
            metric=["mod_chi2"],
            external_priors_penalty=None,
            **fit_octant
        )

In [ ]:
ih_asi_ICU = best_fit_info_ih.detailed_metric_info[0]['mod_chi2']['maps']['total']
ih_asi_DC = best_fit_info_ih.detailed_metric_info[1]['mod_chi2']['maps']['total']
ih_prior = sum(best_fit_info_ih.detailed_metric_info[0]['mod_chi2']['priors'])

ih_asi = best_fit_info_ih.metric_val
print(ih_asi, np.sqrt(ih_asi))

In [ ]:
sens_ICU = calc_sens(nh_asi_ICU+nh_prior, ih_asi_ICU+ih_prior)
sens_DC = calc_sens(nh_asi_DC+nh_prior, ih_asi_DC+ih_prior)

sens = calc_sens(nh_asi, ih_asi)
sens

In [ ]:
plt.scatter(range(3), [sens_DC, sens_ICU, sens])
plt.title('NO = True, no HS or new systematics')
plt.xlim(-0.5,2.5)
plt.xticks(range(3), ['OscNext (12yr)', 'Upgrade(d) (3yr)', 'Combined'])
plt.ylim(0, 3.5)
plt.ylabel(r'$\sigma_{NMO}$')

#plt.savefig('../plots/NMO_dynedge.png')

pseudo experiments

In [ ]:
%%time

nh_dist, ih_dist = [], []
for i in range(10):
    best_fit_info_nh_nh = analysis.fit_recursively(
                data_dist=fake_data_nh.fluctuate(method='poisson', random_state=i),
                hypo_maker=template_maker_NO,
                metric=["mod_chi2"],
                external_priors_penalty=None,
                **fit_octant
            )
    best_fit_info_nh_ih = analysis.fit_recursively(
                data_dist=fake_data_ih.fluctuate(method='poisson', random_state=i),
                hypo_maker=template_maker_NO,
                metric=["mod_chi2"],
                external_priors_penalty=None,
                **fit_octant
            )

    best_fit_info_ih_nh = analysis.fit_recursively(
                data_dist=fake_data_nh.fluctuate(method='poisson', random_state=i),
                hypo_maker=template_maker_IO,
                metric=["mod_chi2"],
                external_priors_penalty=None,
                **fit_octant
            )
    best_fit_info_ih_ih = analysis.fit_recursively(
                data_dist=fake_data_ih.fluctuate(method='poisson', random_state=i),
                hypo_maker=template_maker_IO,
                metric=["mod_chi2"],
                external_priors_penalty=None,
                **fit_octant
            )
    
    nh_dist.append(best_fit_info_nh_nh.metric_val - best_fit_info_ih_nh.metric_val)
    ih_dist.append(best_fit_info_nh_ih.metric_val - best_fit_info_ih_ih.metric_val)

In [ ]:
plt.hist(nh_dist, label=r'Pseudo$_{NO}$', histtype='step', linewidth=3)
plt.hist(ih_dist, label=r'Pseudo$_{IO}$', histtype='step', linewidth=3)
plt.axvline(-nh_asi, label=r'Asimov$_{NO}$')
plt.axvline(ih_asi, color='tab:orange', label=r'Asimov$_{IO}$')

#plt.title('')
plt.legend()
plt.xlabel(r'$\chi^{2}_{NO}-\chi^{2}_{IO}$')
plt.ylabel('#pseudo experiments')

#plt.savefig('../plots/NMO_PE_dynedge', bbox_inches='tight')